# Step 0 — Does ESM-2 encode the *alternate* fold of fold-switching proteins?

**Question.** Fold switchers have one sequence but two dominant native structures. AlphaFold2 can sometimes
recover both by subsampling the MSA; ESM-based models (ESMFold) have no MSA to subsample and are reported to
collapse to a single fold. This notebook tests, before building anything downstream, whether the *signal for
the alternate fold is present at all* inside ESM-2 — in its attention maps and in its categorical Jacobian.

**Design (per fold switcher).** One sequence, two PDB structures → two contact maps. Restrict to residues
resolved in **both** structures, then split pairs by `|i-j| >= 6` into:

| set | meaning |
|---|---|
| `shared` | contact in both folds |
| `f1u` | contact only in fold 1 |
| `f2u` | contact only in fold 2 |
| `bg` | contact in neither |

For every predictor — each layer×head of ESM-2 attention (symmetrized + APC), the **categorical Jacobian**,
and the built-in contact head as a supervised reference — we score two things with AUROC vs a shuffled-label null:

1. **Recovery** (`f2u_recovery`): positives = `f2u`, negatives = `bg`. Does it rank alternate-fold contacts above non-contacts?
2. **Discrimination** (`f2u_vs_f1u`): positives = `f2u`, negatives = `f1u`. Among the *differential* contacts, does it favor fold 2 **in the right direction**? (0.5 = cannot tell the two folds apart — this is the decisive test.)

`shared_recovery` is the sanity check ESM-2 should ace. We combine per-protein z-scores with Stouffer's method.

**Verdict logic.** If some head (or the Jacobian) has `f2u_vs_f1u` significantly > 0.5 across proteins →
the alternate fold *is* encoded → Approach 1 has a foothold. If everything hovers at 0.5 → ESM-2 does not
encode it → a clean, publishable negative result. Either way you have a project.

> **Before a real run:** verify the PDB ids / chain ids in the config cell (I seeded a template set). The
> pipeline prints a sequence-identity warning for each structure pair; low identity means the two PDBs are not
> the same construct and should be swapped. To scale to the full **Porter & Looger** set, just extend
> `FOLD_SWITCHERS` with one row per pair — the alignment step handles numbering/missing residues for you.

In [ ]:
# Run once per environment if needed.
# !pip install -q torch transformers biotite scikit-learn scipy matplotlib numpy pandas tqdm

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import torch
warnings.filterwarnings("ignore")

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME  = "facebook/esm2_t33_650M_UR50D"   # 33 layers x 20 heads; swap for a smaller/larger variant
MIN_SEP     = 6        # |i-j| >= MIN_SEP for a valid pair
CONTACT_A   = 8.0      # CB-CB (CA for GLY) distance threshold, Angstrom
RUN_JACOBIAN= True     # categorical Jacobian is the slow part; toggle off for a fast pass
MAX_LEN_JAC = 400      # skip the Jacobian above this length to bound runtime
QUICK_TEST  = False    # True -> only the first protein, Jacobian off (smoke test)
CACHE_DIR   = "pdb_cache"; os.makedirs(CACHE_DIR, exist_ok=True)
print("device:", DEVICE)

## 1. Fold-switcher set

Each entry is one sequence with two conformations. Mark the ground/dominant state as `fold1` and the
alternate as `fold2` when known (orientation only affects which direction counts as "the alternate fold").
You only need PDB id + chain per conformation; the pipeline aligns the two structures automatically.

In [ ]:
FOLD_SWITCHERS = [
    dict(name="XCL1_lymphotactin",
         fold1=dict(pdb="1j9o", chain="A", label="Ltn10_chemokine"),
         fold2=dict(pdb="2jp1", chain="A", label="Ltn40_dimer_betasandwich"),
         note="Canonical metamorphic protein; same ~93-aa human XCL1 sequence."),
    dict(name="Mad2",
         fold1=dict(pdb="1duj", chain="A", label="O-Mad2_open"),
         fold2=dict(pdb="1s2h", chain="A", label="C-Mad2_closed"),
         note="Spindle checkpoint; open vs closed Mad2, same sequence."),
    dict(name="CLIC1",
         fold1=dict(pdb="1k0m", chain="A", label="soluble_GST-like"),
         fold2=dict(pdb="1rk4", chain="A", label="oxidized_dimer"),
         note="Chloride intracellular channel; redox-driven switch."),
    dict(name="RfaH",
         fold1=dict(pdb="2oug", chain="A", label="autoinhibited_CTD_alpha"),
         fold2=dict(pdb="2lcl", chain="A", label="active_CTD_beta"),
         note="2oug is full-length (CTD alpha-helical); 2lcl is the isolated CTD (beta). Only overlapping CTD residues get scored."),
    dict(name="KaiB",
         fold1=dict(pdb="2qke", chain="A", label="ground_state_FS"),
         fold2=dict(pdb="5jyt", chain="B", label="fold-switched_active"),
         note="CHECK the identity warning: 2qke/5jyt may differ by organism. Replace with a matched pair if identity is low."),
    dict(name="Selecase",
         fold1=dict(pdb="4qhf", chain="A", label="monomer"),
         fold2=dict(pdb="4qhh", chain="A", label="alt_oligomer"),
         note="Metamorphic metalloprotease from M. jannaschii."),
]
print(len(FOLD_SWITCHERS), "fold switchers configured")

## 2. Structures → split contact maps

For each conformation we take one representative coordinate per residue (CB, or CA for glycine / missing CB),
build a contact map, then re-index both folds onto a common **reference sequence** via global alignment.
This absorbs differing residue numbering and missing residues. All labels are restricted to residues
observed in **both** structures (`co` mask), so "unique" means genuinely present-but-not-in-contact, not
merely unresolved.

In [ ]:
import biotite.structure as struc
import biotite.structure.io.pdbx as pdbx
from biotite.database import rcsb
from biotite.sequence import ProteinSequence
from biotite.sequence.align import SubstitutionMatrix, align_optimal

AA20 = "ACDEFGHIKLMNPQRSTVWY"

def fetch_chain(pdb_id, chain):
    path  = rcsb.fetch(pdb_id.lower(), "bcif", CACHE_DIR)
    cif   = pdbx.BinaryCIFFile.read(path)
    atoms = pdbx.get_structure(cif, model=1)          # author chain/res ids by default
    atoms = atoms[struc.filter_amino_acids(atoms)]
    atoms = atoms[atoms.chain_id == chain]
    if atoms.array_length() == 0:
        raise ValueError(f"{pdb_id} chain {chain}: no protein atoms (check the chain id)")
    return atoms

def chain_to_residues(atoms):
    seq, coords = [], []
    for res in struc.residue_iter(atoms):
        cb = res[res.atom_name == "CB"]
        ca = res[res.atom_name == "CA"]
        if   cb.array_length() > 0: c = cb.coord[0]
        elif ca.array_length() > 0: c = ca.coord[0]
        else: continue
        try:    aa = ProteinSequence.convert_letter_3to1(res.res_name[0])
        except Exception: aa = "X"
        seq.append(aa); coords.append(c)
    return "".join(seq), np.asarray(coords, float)

def contact_map(coords):
    d = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=-1)
    return d < CONTACT_A

def sanitize(s):
    return "".join(a if a in AA20 else "X" for a in s)

def map_to_ref(obs_seq, ref_seq):
    # returns {obs_index -> ref_index} and the alignment identity over mapped positions
    mat = SubstitutionMatrix.std_protein_matrix()
    ali = align_optimal(ProteinSequence(sanitize(obs_seq)),
                        ProteinSequence(sanitize(ref_seq)),
                        mat, gap_penalty=(-10, -1), terminal_penalty=False)[0]
    m = {int(o): int(r) for o, r in ali.trace if o != -1 and r != -1}
    o_s, r_s = sanitize(obs_seq), sanitize(ref_seq)
    ident = float(np.mean([o_s[o] == r_s[r] for o, r in m.items()])) if m else 0.0
    return m, ident

In [ ]:
def build_protein(entry):
    s1, c1 = chain_to_residues(fetch_chain(entry["fold1"]["pdb"], entry["fold1"]["chain"]))
    s2, c2 = chain_to_residues(fetch_chain(entry["fold2"]["pdb"], entry["fold2"]["chain"]))
    ref = entry.get("sequence") or (s1 if len(s1) >= len(s2) else s2)
    ref = sanitize(ref); L = len(ref)
    m1, id1 = map_to_ref(s1, ref)
    m2, id2 = map_to_ref(s2, ref)
    cm1, cm2 = contact_map(c1), contact_map(c2)
    C1 = np.zeros((L, L), bool); C2 = np.zeros((L, L), bool)
    present1 = np.zeros(L, bool); present2 = np.zeros(L, bool)
    src1 = np.array(sorted(m1))
    if len(src1):
        dst1 = np.array([m1[i] for i in src1])
        C1[np.ix_(dst1, dst1)] = cm1[np.ix_(src1, src1)]; present1[dst1] = True
    src2 = np.array(sorted(m2))
    if len(src2):
        dst2 = np.array([m2[i] for i in src2])
        C2[np.ix_(dst2, dst2)] = cm2[np.ix_(src2, src2)]; present2[dst2] = True
    return dict(name=entry["name"], seq=ref, L=L, C1=C1, C2=C2,
                co=present1 & present2, id1=id1, id2=id2,
                labels=(entry["fold1"]["label"], entry["fold2"]["label"]))

def pair_masks(P):
    L, co = P["L"], P["co"]
    iu, ju = np.triu_indices(L, k=MIN_SEP)
    keep = co[iu] & co[ju]
    iu, ju = iu[keep], ju[keep]
    c1 = P["C1"][iu, ju]; c2 = P["C2"][iu, ju]
    return dict(i=iu, j=ju,
                shared=c1 & c2, f1u=c1 & ~c2, f2u=c2 & ~c1, bg=~c1 & ~c2)

## 3. ESM-2 predictors

**Attention** (`[layers, heads, L, L]`): symmetrized and APC-corrected per head, following Rao et al.'s
unsupervised contact work. **Categorical Jacobian**: for each position we substitute every amino acid, read
how all other positions' logits respond, center over the input amino acid, and take the Frobenius norm — an
MSA-free coupling map. **Contact head**: ESM-2's own supervised readout, as an upper-reference.

In [ ]:
from transformers import AutoTokenizer, EsmForMaskedLM
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = EsmForMaskedLM.from_pretrained(MODEL_NAME, output_attentions=True).to(DEVICE).eval()
N_LAYERS = model.config.num_hidden_layers
N_HEADS  = model.config.num_attention_heads
print(f"{MODEL_NAME}: {N_LAYERS} layers x {N_HEADS} heads = {N_LAYERS*N_HEADS} attention predictors")

In [ ]:
def apc(x):
    # average product correction; operates on the last two axes
    s1 = x.sum(-1, keepdims=True)
    s2 = x.sum(-2, keepdims=True)
    s  = x.sum((-1, -2), keepdims=True) + 1e-8
    return x - s1 * s2 / s

@torch.no_grad()
def head_predictors(seq):
    enc = tokenizer(seq, return_tensors="pt").to(DEVICE)
    out = model(**enc, output_attentions=True)
    att = torch.stack(out.attentions)[:, 0]           # [layers, heads, L+2, L+2]
    att = att[:, :, 1:-1, 1:-1]                        # strip <cls>/<eos>
    att = att + att.transpose(-1, -2)                  # symmetrize
    att = att.float().cpu().numpy()
    return apc(att)                                    # [layers, heads, L, L]

In [ ]:
@torch.no_grad()
def categorical_jacobian(seq, batch=64):
    vocab  = tokenizer.get_vocab()
    in_ids = [vocab[a] for a in AA20]
    base   = tokenizer(seq, return_tensors="pt").input_ids[0]   # L+2
    L = len(seq)
    variants = []
    for i in range(L):
        for aid in in_ids:
            v = base.clone(); v[i + 1] = aid; variants.append(v)
    variants = torch.stack(variants)                             # [20L, L+2]
    out_ids  = torch.tensor(in_ids)
    logits   = np.empty((20 * L, L, 20), np.float32)
    for s in range(0, variants.shape[0], batch):
        chunk = variants[s:s + batch].to(DEVICE)
        lg = model(chunk, output_attentions=False).logits[:, 1:L + 1, :]
        logits[s:s + chunk.shape[0]] = lg[:, :, out_ids].float().cpu().numpy()
    J = logits.reshape(L, 20, L, 20)        # [i, a, j, b]
    J = J - J.mean(axis=1, keepdims=True)   # center over input amino acid
    S = np.sqrt((J ** 2).sum(axis=(1, 3)))  # [L, L]
    S = 0.5 * (S + S.T)
    return apc(S[None, None])[0, 0]

@torch.no_grad()
def esm_contact_head(seq):
    enc = tokenizer(seq, return_tensors="pt").to(DEVICE)
    for fn in (getattr(model, "predict_contacts", None),
               getattr(getattr(model, "esm", None), "predict_contacts", None)):
        if fn is None: continue
        try:    return fn(enc["input_ids"], enc["attention_mask"])[0].float().cpu().numpy()
        except Exception: continue
    return None

## 4. Scoring vs a shuffled-label null

AUROC asks: does the predictor rank positive pairs above negative pairs? The **shuffled-label null** is
AUROC under random ranking, whose exact analytic form is the Mann-Whitney U test — so we use that directly
(fast enough for all 660 heads). A permutation version is provided below if you want to verify.

In [ ]:
from scipy.stats import mannwhitneyu

def auc_vs_null(v, pos, neg):
    npos, nneg = int(pos.sum()), int(neg.sum())
    if npos == 0 or nneg == 0:
        return np.nan, np.nan, np.nan
    U, p = mannwhitneyu(v[pos], v[neg], alternative="greater")
    auc  = U / (npos * nneg)
    mu   = npos * nneg / 2.0
    sd   = np.sqrt(npos * nneg * (npos + nneg + 1) / 12.0)
    z    = (U - mu) / (sd + 1e-9)
    return auc, z, p

# Optional explicit permutation null (slower) — same answer as the MWU form above.
def auc_vs_null_perm(v, pos, neg, n_perm=200, seed=0):
    from sklearn.metrics import roc_auc_score
    npos, nneg = int(pos.sum()), int(neg.sum())
    if npos == 0 or nneg == 0: return np.nan, np.nan, np.nan
    y    = np.r_[np.ones(npos), np.zeros(nneg)].astype(bool)
    vals = np.r_[v[pos], v[neg]]
    obs  = roc_auc_score(y, vals)
    rng  = np.random.default_rng(seed)
    null = np.array([roc_auc_score(rng.permutation(y), vals) for _ in range(n_perm)])
    z    = (obs - null.mean()) / (null.std() + 1e-9)
    return obs, z, (np.sum(null >= obs) + 1) / (n_perm + 1)

In [ ]:
def predictor_iter(P):
    seq = P["seq"]
    att = head_predictors(seq)
    for l in range(att.shape[0]):
        for h in range(att.shape[1]):
            yield f"attn_L{l:02d}_H{h:02d}", att[l, h]
    if RUN_JACOBIAN and not QUICK_TEST and P["L"] <= MAX_LEN_JAC:
        yield "categorical_jacobian", categorical_jacobian(seq)
    ch = esm_contact_head(seq)
    if ch is not None:
        yield "esm_contact_head", ch

METRICS = [("f2u_recovery",   "f2u", "bg"),
           ("f1u_recovery",   "f1u", "bg"),
           ("shared_recovery","shared", "bg"),
           ("f2u_vs_f1u",     "f2u", "f1u")]

entries   = FOLD_SWITCHERS[:1] if QUICK_TEST else FOLD_SWITCHERS
proteins  = {}
rows      = []
for e in entries:
    try:
        P = build_protein(e)
    except Exception as ex:
        print("SKIP", e["name"], "->", ex); continue
    proteins[e["name"]] = P
    M = pair_masks(P)
    flag = "  <-- LOW IDENTITY, check this pair" if min(P["id1"], P["id2"]) < 0.9 else ""
    print(f"{e['name']:>18s}: L={P['L']:3d} id=({P['id1']:.2f},{P['id2']:.2f}) co={int(P['co'].sum()):3d} | "
          f"shared={int(M['shared'].sum()):4d} f1u={int(M['f1u'].sum()):3d} "
          f"f2u={int(M['f2u'].sum()):3d} bg={int(M['bg'].sum()):5d}{flag}")
    if M["f2u"].sum() == 0 or M["f1u"].sum() == 0:
        print(f"    [warn] little differential signal for {e['name']}")
    ii, jj = M["i"], M["j"]
    for pname, S in predictor_iter(P):
        v = S[ii, jj]
        for tag, pk, nk in METRICS:
            auc, z, p = auc_vs_null(v, M[pk], M[nk])
            rows.append(dict(protein=e["name"], predictor=pname, metric=tag, auc=auc, z=z, p=p))

res = pd.DataFrame(rows)
res.to_csv("foldswitch_step0_results.csv", index=False)
print("\nresults:", res.shape, "-> foldswitch_step0_results.csv")

## 5. Where in the network is the signal? (layer × head maps)

Mean z-score across proteins for each attention head. `shared_recovery` should light up (ESM-2 knows the
dominant fold). The question is whether anything lights up for `f2u_recovery` and, decisively, `f2u_vs_f1u`.

In [ ]:
import matplotlib.pyplot as plt

def head_grid(metric, value="z"):
    d = res[(res.metric == metric) & res.predictor.str.startswith("attn_")].copy()
    d["layer"] = d.predictor.str.extract(r"L(\d+)_")[0].astype(int)
    d["head"]  = d.predictor.str.extract(r"H(\d+)")[0].astype(int)
    g = d.groupby(["layer", "head"])[value].mean()
    grid = np.full((N_LAYERS, N_HEADS), np.nan)
    for (l, h), val in g.items():
        grid[l, h] = val
    return grid

fig, axes = plt.subplots(1, 4, figsize=(17, 7), sharey=True)
for ax, metric in zip(axes, ["shared_recovery", "f2u_recovery", "f1u_recovery", "f2u_vs_f1u"]):
    grid = head_grid(metric)
    lim  = np.nanmax(np.abs(grid)) if np.isfinite(grid).any() else 1.0
    im = ax.imshow(grid, aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.set_title(metric); ax.set_xlabel("head")
    fig.colorbar(im, ax=ax, fraction=0.046)
axes[0].set_ylabel("layer")
plt.tight_layout(); plt.show()

## 6. Verdict

Stouffer-combine per-protein z-scores for each predictor. A significantly positive **`f2u_vs_f1u`** z means
some predictor favors the alternate fold *in the right direction* across proteins — the alternate fold is
encoded. If the top `f2u_vs_f1u` predictors sit near z=0, ESM-2 does not encode it (negative result).

In [ ]:
from scipy.stats import norm

def combine(metric, topn=12):
    d = res[res.metric == metric].dropna(subset=["z"])
    g = d.groupby("predictor").agg(
            mean_auc=("auc", "mean"),
            stouffer_z=("z", lambda x: x.sum() / np.sqrt(len(x))),
            n=("z", "size"))
    g["p_combined"] = norm.sf(g["stouffer_z"])
    return g.sort_values("stouffer_z", ascending=False).head(topn)

print("=== shared_recovery (sanity: ESM-2 should ace the dominant fold) ===")
display(combine("shared_recovery", 5))
print("=== f2u_recovery: ranks alternate-fold contacts above non-contacts? ===")
display(combine("f2u_recovery"))
print("=== f2u_vs_f1u: favors fold 2 over fold 1 among differential contacts? (THE test) ===")
display(combine("f2u_vs_f1u"))

In [ ]:
rec  = combine("f2u_recovery", 1).iloc[0]
disc = combine("f2u_vs_f1u", 1).iloc[0]
print(f"Top f2u_recovery : {rec.name:>22s}  auc={rec.mean_auc:.3f}  Stouffer z={rec.stouffer_z:6.2f}  p={rec.p_combined:.2e}")
print(f"Top f2u_vs_f1u   : {disc.name:>22s}  auc={disc.mean_auc:.3f}  Stouffer z={disc.stouffer_z:6.2f}  p={disc.p_combined:.2e}")
print()
if disc.stouffer_z > 3:
    print("VERDICT: a predictor discriminates the alternate fold in the right direction across proteins.")
    print("         The alternate fold IS (partly) encoded -> Approach 1 has a foothold; inspect this head/map.")
elif rec.stouffer_z > 3:
    print("VERDICT: alternate-fold contacts are recovered above chance, but not clearly ABOVE fold-1 contacts.")
    print("         Weak/ambiguous signal -> worth probing but not a clean win.")
else:
    print("VERDICT: no predictor beats the shuffled null for the alternate fold -> ESM-2 does not encode it.")
    print("         This is a clean, publishable negative result.")

## 7. Inspect one protein

Grey = shared contacts, blue = fold-1-unique, red = fold-2-unique (on the common reference indexing).
Overlay the top `f2u_vs_f1u` head to see *where* it is (or isn't) looking.

In [ ]:
def show_protein(name):
    P, M = proteins[name], pair_masks(proteins[name])
    L = P["L"]; img = np.ones((L, L, 3))
    def paint(mask, color):
        for i, j, m in zip(M["i"], M["j"], mask):
            if m: img[i, j] = color; img[j, i] = color
    paint(M["shared"], [0.6, 0.6, 0.6]); paint(M["f1u"], [0.15, 0.35, 1.0]); paint(M["f2u"], [1.0, 0.2, 0.1])
    plt.figure(figsize=(7, 7)); plt.imshow(img)
    plt.title(f"{name}\ngrey=shared  blue={P['labels'][0]} unique  red={P['labels'][1]} unique")
    plt.xlabel("residue"); plt.ylabel("residue"); plt.tight_layout(); plt.show()

if proteins:
    show_protein(list(proteins)[0])

## 8. Notes & next steps

- **Verify the config.** The seeded PDB/chain ids are a template. The run prints an identity warning per
  pair (`id=(...)`); a value well below ~0.9 means the two structures are not the same construct — fix the
  entry (KaiB's `2qke`/`5jyt` is the most likely to need a matched pair).
- **Full Porter & Looger set.** Extend `FOLD_SWITCHERS` from their supplementary table; nothing else changes.
  With ~90 pairs the Stouffer test gets real power.
- **Orientation.** `f2u_vs_f1u` is directional. If you don't know which fold ESM "prefers", also look at
  `f1u_vs_f2u` (just swap, or read `-z`); the honest question is whether *either* non-dominant fold is recovered.
- **Cost.** Attention scoring is seconds per protein. The categorical Jacobian is ~`20*L` forward passes —
  the slow part; cap it with `MAX_LEN_JAC` or set `RUN_JACOBIAN=False` for a first pass.
- **If signal exists (Approach 1):** the implicated heads/Jacobian couplings become features for recovering
  the alternate structure. **If not:** the negative result is the paper — ESM-2 collapses to one fold and
  attention/Jacobian interpretability cannot recover the other.

## 9. Diagnostics — is the discrimination signal real, or a confound?

The auto-verdict flags `f2u_vs_f1u` as significant, but AUC ~0.60 with huge pair counts is *statistical*
significance, not *effect size*. Two confounds must be ruled out before 0.60 means "ESM encodes the alternate fold":

1. **Sequence separation.** Short-range contacts are trivially predictable. If fold-2-unique contacts have a
   different `|i-j|` distribution than fold-1-unique, any contact predictor scores >0.5 on `f2u_vs_f1u` for free.
   We plot the distributions, add a **separation-only baseline** (score = -|i-j|), and recompute discrimination
   **stratified by separation**. If the controlled AUC collapses to ~0.5, the signal was just separation.
2. **Arbitrary orientation.** `fold2` is just the 2nd PDB I listed. We re-orient per protein by which fold ESM
   actually prefers, then ask whether any head recovers the *suppressed* fold.

Plus a **per-protein** breakdown, since n=5 + Stouffer can be driven by one protein (e.g. a low-identity pair).

In [ ]:
from scipy.stats import mannwhitneyu

_att_cache = {}
def score_matrix(P, name):
    # returns the LxL predictor score matrix for a protein, caching attention
    if name.startswith("attn_"):
        if P["name"] not in _att_cache:
            _att_cache[P["name"]] = head_predictors(P["seq"])
        l = int(name[6:8]); h = int(name[10:12])
        return _att_cache[P["name"]][l, h]
    if name == "categorical_jacobian": return categorical_jacobian(P["seq"])
    if name == "esm_contact_head":     return esm_contact_head(P["seq"])
    raise ValueError(name)

def auc_dir(v, pos, neg):
    if pos.sum() < 5 or neg.sum() < 5: return np.nan
    U, _ = mannwhitneyu(v[pos], v[neg], alternative="greater")
    return U / (int(pos.sum()) * int(neg.sum()))

SEP_BINS = np.array([6, 10, 16, 24, 36, 54, 80, 10000])
def sep_stratified_auc(v, sep, pos, neg, bins=SEP_BINS):
    # AUC of pos-vs-neg computed within separation bins, then pair-count weighted
    aucs, ws = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        b = (sep >= lo) & (sep < hi)
        a = auc_dir(v, pos & b, neg & b)
        if not np.isnan(a):
            aucs.append(a); ws.append(int((pos & b).sum()) + int((neg & b).sum()))
    return np.average(aucs, weights=ws) if aucs else np.nan

### 9a. Separation distributions of the two folds' unique contacts

If these two histograms differ, `f2u_vs_f1u` is confounded by range.

In [ ]:
import matplotlib.pyplot as plt
f1u_sep, f2u_sep = [], []
for name, P in proteins.items():
    M = pair_masks(P); sep = np.abs(M["i"] - M["j"])
    f1u_sep.append(sep[M["f1u"]]); f2u_sep.append(sep[M["f2u"]])
f1u_sep = np.concatenate(f1u_sep); f2u_sep = np.concatenate(f2u_sep)

plt.figure(figsize=(7, 4))
bins = np.linspace(6, max(f1u_sep.max(), f2u_sep.max()), 30)
plt.hist(f1u_sep, bins=bins, alpha=0.6, density=True, label=f"fold1-unique (median {np.median(f1u_sep):.0f})")
plt.hist(f2u_sep, bins=bins, alpha=0.6, density=True, label=f"fold2-unique (median {np.median(f2u_sep):.0f})")
plt.xlabel("sequence separation |i-j|"); plt.ylabel("density")
plt.title("Separation distributions of fold-unique contacts"); plt.legend(); plt.tight_layout(); plt.show()
print(f"pooled: f1u median |i-j| = {np.median(f1u_sep):.1f}, f2u median |i-j| = {np.median(f2u_sep):.1f}")

### 9b. Raw vs separation-controlled discrimination + the separation-only baseline

`sep_baseline` is score = -|i-j| (pure "closer pairs rank higher"). Its **raw** `f2u_vs_f1u` AUC is how much
discrimination you get for *free* from the range distribution. For each real predictor, compare **raw** to
**sep-controlled**: if controlled drops toward the baseline / toward 0.5, the apparent signal was separation.

In [ ]:
def discrimination_report(name):
    raw, ctrl = [], []
    for _, P in proteins.items():
        M = pair_masks(P); sep = np.abs(M["i"] - M["j"])
        v = -sep.astype(float) if name == "sep_baseline" else score_matrix(P, name)[M["i"], M["j"]]
        r = auc_dir(v, M["f2u"], M["f1u"]);
        if not np.isnan(r): raw.append(r)
        c = sep_stratified_auc(v, sep, M["f2u"], M["f1u"])
        if not np.isnan(c): ctrl.append(c)
    return np.mean(raw) if raw else np.nan, np.mean(ctrl) if ctrl else np.nan

top_disc = list(combine("f2u_vs_f1u", 8).index)
rows = []
for name in ["sep_baseline"] + top_disc:
    r, c = discrimination_report(name)
    rows.append(dict(predictor=name, raw_auc=r, sep_controlled_auc=c, drop=r - c))
rep = pd.DataFrame(rows).set_index("predictor")
print("0.50 = chance. 'raw' includes separation; 'sep_controlled' removes it.")
print("If a predictor's sep_controlled_auc ~ the sep_baseline row, its signal WAS separation.\n")
display(rep.round(3))

### 9c. Per-protein breakdown of the top discriminating heads

Is the pooled 0.60 consistent across all 5 proteins, or driven by one (possibly a low-identity pair)?

In [ ]:
rows = []
for name in top_disc:
    row = {"predictor": name}
    for pname, P in proteins.items():
        M = pair_masks(P)
        v = score_matrix(P, name)[M["i"], M["j"]]
        row[pname] = auc_dir(v, M["f2u"], M["f1u"])
    rows.append(row)
per_prot = pd.DataFrame(rows).set_index("predictor")
per_prot["mean"] = per_prot.mean(axis=1)
display(per_prot.round(3))
print("A real effect is ~consistently >0.5 across columns. One hot column + rest ~0.5 = an artifact / single-protein effect.")

### 9d. ESM-preference orientation — does any head recover the *suppressed* fold?

Orientation so far is arbitrary (PDB listing order). Here we define, per protein, the fold ESM **prefers** as
the one its contact head recovers better, call the other the **suppressed** fold, and ask whether any attention
head recovers the suppressed-fold-unique contacts above the preferred-fold-unique ones — the biologically
meaningful "can we get the fold ESM collapses away from" question. This re-orients every protein consistently.

In [ ]:
# which fold does ESM prefer, per protein? (contact head: recovery of f1u vs f2u over background)
orient = {}
for pname, P in proteins.items():
    M = pair_masks(P); ch = score_matrix(P, "esm_contact_head")[M["i"], M["j"]]
    a1 = auc_dir(ch, M["f1u"], M["bg"]); a2 = auc_dir(ch, M["f2u"], M["bg"])
    orient[pname] = ("fold1", "fold2") if a1 >= a2 else ("fold2", "fold1")  # (preferred, suppressed)
    print(f"{pname:>18s}: contact-head recovers f1u@{a1:.3f} f2u@{a2:.3f} -> prefers {orient[pname][0]}")

def suppressed_auc(name):
    vals = []
    for pname, P in proteins.items():
        M = pair_masks(P); pref, sup = orient[pname]
        pos = M["f2u"] if sup == "fold2" else M["f1u"]
        neg = M["f2u"] if pref == "fold2" else M["f1u"]
        v = score_matrix(P, name)[M["i"], M["j"]]
        a = auc_dir(v, pos, neg)
        if not np.isnan(a): vals.append(a)
    return np.mean(vals) if vals else np.nan

# scan every attention head for suppressed-fold recovery, oriented consistently
heads = sorted({p for p in res.predictor.unique() if p.startswith("attn_")})
scan = pd.DataFrame({"predictor": heads,
                     "suppressed_vs_preferred_auc": [suppressed_auc(h) for h in heads]})
scan = scan.sort_values("suppressed_vs_preferred_auc", ascending=False)
print("\nTop heads at recovering the SUPPRESSED fold (oriented by ESM preference); 0.50 = chance:")
display(scan.head(10).round(3))
print(f"\nbest suppressed-fold head AUC = {scan.iloc[0,1]:.3f}  "
      f"(sep-baseline suppressed AUC shown below for reference)")

### Read-out

- **Separation plot + 9b:** if `sep_controlled_auc` for the top heads sits near the `sep_baseline` row (and near 0.5),
  the discrimination was a range artifact -> **negative result** (ESM-2 does not encode the alternate fold).
- **9c:** a real effect is consistently >0.5 across proteins. One hot column = drop that protein and re-check.
- **9d:** the honest headline number. If the best suppressed-fold head stays ~0.5 after orientation + separation
  control, that is your clean negative result. If one or two heads hold ~0.6-0.7 *after* both controls and across
  proteins, that is a genuine foothold worth the interpretability write-up.

Whichever way it lands, 9b+9d is the figure/claim that survives review — not the raw Stouffer z.